# 라이브러리 임포트

In [ ]:
from gensim.models import Word2Vec, FastText
import torch
import torch.nn as nn
import numpy as np
import wandb
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize
import matplotlib.pyplot as plt
import re
from nltk.corpus import stopwords
import nltk
from sklearn.datasets import fetch_20newsgroups
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
import pandas as pd




nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')


nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')

## GPU 설정

In [ ]:
# GPU 설정
device = torch.device("mps" if torch.cuda.is_available() else "cpu")
device = torch.device("mps")
print(device)

# 데이터 확인

## 데이터 불러오기

In [ ]:
# 훈련 데이터와 테스트 데이터를 모두 가져옴
# news_data는 data, target으로 구성됨
news_data = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))

In [ ]:
texts = news_data.data
texts[20]

In [ ]:
labels = news_data.target
labels

In [ ]:
label_names = news_data.target_names
label_names

In [ ]:
print(f"Unique labels: {set(labels)}")
# 0 ~ 19까지 20가지 뉴스 카테고리가 있음.
# 크게 컴퓨터, 스포츠, 과학, 정치... -> 세부

In [ ]:
print(f"Number of documents: {len(texts)}")
# 18846개. 0~18845개 있음
print(f"Sample document: \n{texts[5553]}")
print(f"Label name: {news_data.target_names[labels[5553]]}")

# 데이터 전처리

## SToP words 지우기

In [ ]:
stop_words = stopwords.words('english')
print(f"{len(stop_words)}개\n{stop_words[:8]}")

In [ ]:
print(len(texts))
print(len(labels))
print(labels[:10])

In [ ]:
# 로 나누고
train_inputs, test_inputs, train_targets, test_targets= train_test_split(texts, labels, test_size=0.2, random_state=42, stratify=labels) # 걍해도 나쁠 거 없을듯

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(word):
    """단어의 품사를 WordNet 표기로 변환"""
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB,
                "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)



def clean_text(text):
    text = text.lower() # lowecase로 변환
    # text에서 알파벳,공백을 제외하고 '' 으로 치환해라
    text = re.sub(r'[^a-zA-Z\s]', '', text)# (패턴, 바꿀문자열, 대상문자열)
    stop_words = set(stopwords.words('english'))
    tokens = text.split() # 공백 기준으로 나눠서 리스트에 들어감
    # strip은 알아서 됨
    # ["i", "like", "coffee"]

    # 제네레이터도 ㅗ딜듯
    # 불용어 제거 + 2글자 이하 제거
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]

    # 표제어 추출
    lemmas = [lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in tokens]
    return ' '.join(lemmas)

train_inputs = [clean_text(text) for text in train_inputs]
test_inputs = [clean_text(text) for text in test_inputs]

print(train_inputs[0])
print("\n")
print(label_names[train_targets[0]])


## 문자열 길이 시각화

In [ ]:
# 메시지 길이 계산 nltk안에 있음

# 먼저 전처리해서 쓸데없는 거 날려버리고 공백 기준으로 리스트 만들어서 개수 세기
train_lengths = [len(word_tokenize(msg)) for msg in train_inputs]
test_lengths = [len(word_tokenize(msg)) for msg in test_inputs]


# 히스토그램 시각화
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.hist(train_lengths, bins=30, alpha=0.7, label='Train Data')
plt.hist(test_lengths, bins=30, alpha=0.7, label='Test Data')
plt.xlabel('word len')
plt.ylabel('Count', rotation=0)
plt.legend()
plt.title('distribution of word lengths')
# 아니 프린트해서 안 보면 6천개 있는줄도 모름

plt.subplot(1, 2, 2)
plt.hist(train_lengths, bins=50, log=True)
plt.title("distribution of word lengths (log-scaled)")
plt.xlabel("Document Length")
plt.ylabel("Count (log scale)")
plt.tight_layout()
plt.show()


plt.figure(figsize=(10, 5))
plt.subplot(1, 1, 1)
plt.boxplot(train_lengths, vert=False)
plt.title("Boxplot of Document Lengths")
plt.xlabel("Document Length")


plt.tight_layout()
plt.show()

## Laebl distributioon 시각화
- 생각보다 균형ㅈㄱ인데? stratified split 필요없을거같음

In [ ]:
label_counts = pd.Series(train_targets).value_counts().sort_index()
plt.figure(figsize=(10,5))
plt.bar(label_counts.index, label_counts.values, color='skyblue', edgecolor='black')
plt.xticks(range(0, 20))  # 라벨 0~19
plt.xlabel('Class Label')
plt.ylabel('Count', rotation=0)
plt.title('Distribution of Training Labels')
plt.grid(linestyle='--', alpha=0.7, axis='y')
plt.show()

In [ ]:
# # stratify X
# x1_train, x1_test, y1_train, y1_test = train_test_split(texts, labels, test_size=0.2, random_state=42)
#
# # stratify O
# x2_train, x2_test, y2_train, y2_test = train_test_split(texts, labels, test_size=0.2, random_state=42, stratify=labels)
#
# print("Without stratify:", pd.Series(y1_train).value_counts().sort_index())
# print("With stratify:", pd.Series(y2_train).value_counts().sort_index())


# 큰 차이 없음

## 전처리
- 불용어, 정규식 처리하고 토큰화한 길이가 0인것(427장)과 iQR Q1(21 단어)를 지우면 너무 에바고 음 얼마나 지우지
- 한 문장에 대충 10단어 정도면 어느 기사인지 알듯... min_len 바꿔서 돌려보기
- 대부분의 데이터가 500단어 미만임
- 보다 큰 것들은 그냥 살리고 max_len을 500해서 앞에서 500단어만 읽으면 될듯 ㅅㅂ 500너무많은데
- 평균이 대충 100, 100단어봐도될듯

In [ ]:
# 구간(bin) 경계값: 0~500, 500~1000, ..., 6500~

print(len(train_lengths)) # 15076개
print(len(test_lengths))

print(f"평균 길이: {np.mean(train_lengths)}")
print(f"최대 길이:{np.max(train_lengths)}")
print(f"길이 표준편차 {np.std(train_lengths)}\n")
# 편차가 ㅈㄴ크네
print(f"최소 길이:{np.min(train_lengths)}")
bins = np.arange(0, 1000, 100)  # 0~6500까지 500단위 구간
hist, edges = np.histogram(train_lengths, bins=bins)

# 출력
for i in range(len(hist)):
    print(f"{int(edges[i]):>4} ~ {int(edges[i+1]):>4} 단어 : {hist[i]}개")

# 마지막 구간에 포함되지 않은 초과치가 있다면 따로 확인
above_max = np.sum(np.array(train_lengths) >= bins[-1])
print(f"≥ {bins[-1]} 단어 : {above_max}개")
zero_count = sum(1 for l in train_lengths if l == 0)
print(f"길이가 0인 문서 개수: {zero_count}")

In [ ]:
# 이미 있는 train_lengths를 사용한다고 가정
bins = list(range(0, 100, 5))  # 0~500 단어를 20단어 단위로
counts, _ = np.histogram(train_lengths, bins=bins)

for i in range(len(bins)-1):
    print(f"{bins[i]:3d} ~ {bins[i+1]-1:3d} 단어 : {counts[i]}개")

print(f"≥ {bins[-1]} 단어 : {(np.array(train_lengths) >= bins[-1]).sum()}개")

print(f"\n최소 길이: {np.min(train_lengths)}")
print(f"최대 길이: {np.max(train_lengths)}")
print(f"평균 길이: {np.mean(train_lengths):.2f}")

In [ ]:
# # numpy array로 변환
# arr = np.array(train_lengths)

# # 사분위수 계산
# q1 = np.percentile(arr, 25)
# q3 = np.percentile(arr, 75)
# iqr = q3 - q1

# # 이상치 기준
# upper_bound = q3 + 1.5 * iqr
# lower_bound = q1 - 1.5 * iqr

# # 이상치 값 추출
# outliers = arr[(arr > upper_bound) | (arr < lower_bound)]

# print(f"📈 1사분위수(Q1): {q1:.2f}")
# print(f"📉 3사분위수(Q3): {q3:.2f}")
# print(f"🧮 IQR (Q3 - Q1): {iqr:.2f}")
# print(f"🔼 상한선: {upper_bound:.2f}")
# print(f"🔽 하한선: {lower_bound:.2f}")
# print(f"⚠️ 이상치 개수: {len(outliers)}개")
# print(f"⚠️ 이상치 중 최대값: {outliers.max() if len(outliers) > 0 else '없음'}")
# print(f"⚠️ 이상치 인덱스: {np.where(arr > upper_bound)[0]}")

In [ ]:
clean_train = [
    (x, y) for x, y, l in zip(train_inputs, train_targets, train_lengths)
    if l >= 11   # 11단어 이상 문서만 유지
]

# 다시 분리
train_inputs, train_targets = zip(*clean_train)
train_inputs = list(train_inputs)
train_targets = list(train_targets)

print(f"11단어 이상 문서 개수: {len(train_inputs)}")

# Word2VEC

## 토큰화

In [ ]:
train_sentences = [word_tokenize(text) for text in train_inputs]
# train_sentences = [text.split() for text in train_sentences]

test_sentences = [word_tokenize(text) for text in test_inputs]
# 나중에 쓸거니깐 제네레이터말고...
print(train_sentences[0]) # 첫번째 문서 토큰화

## word2vec 모델 생성 skip-gram

In [ ]:
# 사전과 임베딩 만들기
word2vec_model_skip_gram = Word2Vec(sentences=train_sentences, vector_size=128, window=8, min_count=3, workers=4, sg=1) # 안에 들어가서 파라미터 확인하기


# sg : {0, 1}, optional Training algorithm: 1 for skip-gram; otherwise CBOW.
# word2vec_model.wv.index_to_key  # ['coffee', 'tea', 'milk', ...]
# word2vec_model.wv["coffee"]  # → array([0.12, -0.47, ..., 0.05])


# 모델의 row:사전 단어의 개수 + padding, 100차원 column
word2vec_matrix = np.zeros((len(word2vec_model_skip_gram.wv) + 1, 128))


# 단어를 키로, idx를 value로 dict으로만 만듦
# {"coffee":1}
word2idx_word2vec_dict = {word: idx + 1 for idx, word in enumerate(word2vec_model_skip_gram.wv.index_to_key)}

# 방금 만든 사전에서 키, 밸류 받아와서 {"coffee":1}
# matrix[1] = [0.1, 0.2....] (coffee의 임베딩 벡터)
for word, idx in word2idx_word2vec_dict.items():
    word2vec_matrix[idx] = word2vec_model_skip_gram.wv[word]


print(sorted(word2vec_model_skip_gram.wv.index_to_key))
# print(word2vec_model_skip_gram.wv.get_vecattr('gunshmmm', 'count'))  # 특정 단어의 등장 횟수

In [ ]:
class TextEmbeddingDataset(Dataset):
    def __init__(self, texts, labels, word2idx, max_len):
#         texts = [
#     "god does not exist",
#     "windows drivers update problem",
#     "the baseball game was great",
#     "nasa launches new satellite",
#     ...
# ]
        self.texts = texts
        self.labels = labels
        self.word2idx = word2idx
#         word2idx = {
#     'the': 1,
#     'cat': 2,
#     'sat': 3,
#     'on': 4,
#     'mat': 5
# }
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = word_tokenize(self.texts[idx]) ## idx번째 꺼내서 띄어쓰기 단위로 토큰화 ["i", "love"]
# word2idx 딕셔너리에서 word라는 단어의 인덱스를 찾고 없다면 0
        encoded = [self.word2idx.get(word, 0) for word in tokens]  # OOV 단어는 0, min_count = 1이라서 Unknonw은 없을듯
        # encoded = [3, 4, 5, ...]

        # 위에서 np.zeros한거는 각 단어별 100차원
        # 여기는 문장별로 길이가 짧으면
        if len(encoded) < self.max_len:     # 이건 패딩용, 최대 길이보다 문장이 짧으면 그 길이만큼 0 append하기
            encoded += [0] * (self.max_len - len(encoded))
        else:
            encoded = encoded[:self.max_len]
        return torch.tensor(encoded, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

In [ ]:
max_len = 200

# Word2Vec Dataset

# 단어 딕셔너리 주고
# train 전체에서 한 문서를 뽑아서 띄어쓰기 단위로 토큰화 하고
# 단어를 딕셔너리에 매핑된 숫자로 치환하고
# 짧은 단어는 패딩해서 내보내기
train_dataset_word2vec = TextEmbeddingDataset(train_inputs, train_targets, word2idx_word2vec_dict, max_len)
test_dataset_word2vec = TextEmbeddingDataset(test_inputs, test_targets, word2idx_word2vec_dict, max_len)

# Word2Vec DataLoader
train_loader_word2vec = DataLoader(train_dataset_word2vec, batch_size=64, shuffle=True)
test_loader_word2vec = DataLoader(test_dataset_word2vec, batch_size=64, shuffle=False)

In [ ]:
from sklearn.metrics import classification_report
def train(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for texts, labels in loader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(texts)  # 출력: (batch_size, 20)
        loss = criterion(outputs, labels)  # 레이블은 정수형 (0~19)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, label_names=None):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for texts, labels in loader:
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)
            predictions = torch.argmax(outputs, dim=1)

            all_preds.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # 전체 accuracy
    accuracy = sum(int(p == t) for p, t in zip(all_preds, all_labels)) / len(all_labels)

    # 클래스별 precision, recall, f1 출력
    report = classification_report(
        all_labels,
        all_preds,
        target_names=label_names if label_names is not None else None,
        digits=4
    )

    print("📊 Classification Report")
    print(report)
    print(f"🎯 Accuracy: {accuracy:.4f}")

    return accuracy

## LSTM 블럭

In [ ]:
class EmbeddingLSTM(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim, output_dim, num_layers=2, dropout=0.3):
        super(EmbeddingLSTM, self).__init__()
        num_embeddings, embedding_dim = embedding_matrix.shape # 행, 열 / 모양은 아까 만든 [len(wv),100]일듯

        # 이미 학습된 임베딩 행렬을 모델 안으로 불러와라
        # matrix가 2차원 리스트 -> 텐서
        self.embedding = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix, dtype=torch.float).to(device), freeze=False)
#         x = torch.tensor([1, 2, 3])  # "coffee", "tea", "milk"
#         embedding(x)
#         [
        #  [0.12, -0.47,  0.05, ...],  # coffee 벡터
        #  [-0.21, 0.35,  0.18, ...],  # tea 벡터
        #  [0.02,  0.11, -0.08, ...]   # milk 벡터
        # ]
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout, bidirectional=False)
        # output은 배치 펄스트니깐 (batch_size, seq_len, hidden_dim)


        # self.bn = nn.BatchNorm1d(hidden_dim)


        self.fc = nn.Linear(hidden_dim, output_dim)


    def forward(self, x):
        embedded = self.embedding(x) # 임베딩 시키고
        # output h1: I에 해당하는 100차원을 넣어서 128차원으로 출력
        # 두번째 셀에 h1 128차원과 like에 해당하는 100차원 넣어서 h2 128차원으로 출력
        _, (hidden, _) = self.lstm(embedded) # output(매 셀 출력), hidden(가장 마지막층), cell
        # 그럼 output의 h1을 두번째 셀로 보내는 건 어딨음
        # 자동으로 넘김
        # hidden[-1] : 마지막 layer의 backward
        hidden = hidden[-1]
        #batchnorm해보기
        # hidden = self.bn(hidden)

        output = self.fc(hidden)
        return output


## GRU block

In [ ]:
class EmbeddingGRU(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim, output_dim, num_layers=1, dropout=0.5):
        super().__init__()
        num_embeddings, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float).to(device),
            freeze=False # 학습하면서 파라미터 바뀜
        )

        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True
        )
        self.layernorm = nn.LayerNorm(hidden_dim * 2)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)                # (batch, seq, emb)
        output, hidden = self.gru(embedded)
        # hidden: (num_layers*2, batch, hidden_dim)

        # 마지막 layer 기준으로 forward/backward 가져오기
        # forward: [last_layer * 2 - 2]
        # backward: [last_layer * 2 - 1]
        last_forward  = hidden[-2]
        last_backward = hidden[-1]

        hidden_cat = torch.cat((last_forward, last_backward), dim=1)


         # LayerNorm 적용
        normed = self.layernorm(hidden_cat)


        return self.fc(normed)

In [ ]:
hidden_dim = 128
output_dim = len(set(labels))

In [ ]:
# model_word2vec = EmbeddingLSTM(word2vec_matrix, hidden_dim, output_dim).to(device)
model_word2vec = EmbeddingGRU(word2vec_matrix, hidden_dim=128, output_dim=20).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_word2vec.parameters(), lr=0.005)
# optimizer = torch.optim.AdamW(
#     model_word2vec.parameters(),
#     lr=0.001,
#     weight_decay=0.01   # ← 이것도 살짝 주면 효과적
# )

# Start a new wandb run to track this script.
wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="yoonwogud-lab",
    # Set the wandb project where this run will be logged.
    project="News20Groups-Word2Vec",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.005,
        "optimizer": "Adam",
        "loss_fn":"cross_entropy",
        "architecture": "Word2Vec-GRU",
        "dataset": "News20Groups-Word2Vec",
        "epochs": 10,
        "vector_size": 128,
        "min_count": 3,
        "sg":"skip-gram",
        "window":8,
        "max_len": 200,
        "batch": 64
    },
)

for epoch in range(10):
    loss = train(model_word2vec, train_loader_word2vec, loss_fn, optimizer)
    accuracy = evaluate(model_word2vec, test_loader_word2vec)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}, Acc: {accuracy:.4f}")
    wandb.log({"Loss": loss, "Acc": accuracy})


wandb.finish()
print(f"Test Accuracy (Word2Vec with LSTM): {accuracy:.4f}")
evaluate(model_word2vec, test_loader_word2vec, label_names)

# 제일 좋은 결과  Acc:0.64 -> Acc:0.7122
- architecture:"Word2Vec-GRU"
- batch:64
- dataset:"News20Groups-Word2Vec"
- epochs:10
- learning_rate:0.005
- loss_fn:"cross_entropy"
- max_len:200
- min_count:3
- optimizer:"Adam"
- sg:"skip-gram"
- vector_size:128
- window:8 
- Dropout:0.5, 근데 layer =1 이니깐 적용안됨
- norm 쓰니깐 아주 좀 올라가긴함...
- 19번 클래스가 좀 어렵긴한가봄 'talk.religion.misc'], 아마 0번이랑 christian같은 비슷한 클래스랑 헷갈린거같음.

- 음 파라미터도 너무 Greedy하게 그냥 잘되는 쪽으로만 바꿔서 특정한 방향성(?)없이 간 거 같네요
- 어텐션올유니드랑 위키독스 본다고 많이 못한거같네요 ㅠ

In [ ]:
for test_sentences, labels in test_loader_word2vec:
    test_sentences, labels = test_sentences.to(device), labels.to(device)
    break
# 예측
model_word2vec.eval()
with torch.no_grad():
    outputs = model_word2vec(test_sentences)
    preds = torch.argmax(outputs, dim=1)
idx = 12

print("=" * 70)
print("📝 원문 문장:")
print(test_inputs[idx])  # <- test_inputs는 texts 리스트라 원문 그대로 볼 수 있음
print("-" * 70)
print("실제 라벨 :", label_names[test_targets[idx]])
print("예측 라벨 :", label_names[preds[idx].item()])
# ㅅㅂ 이걸 틀리네

# FastText

In [ ]:
# FastText 모델 학습
fasttext_model = FastText(sentences=train_sentences, vector_size= 128, window=8, min_count=3, sg=1)
fasttext_matrix = np.zeros((len(fasttext_model.wv) + 1, 128))

word2idx_fasttext = {word: idx + 1 for idx, word in enumerate(fasttext_model.wv.index_to_key)}

for word, idx in word2idx_fasttext.items():
    fasttext_matrix[idx] = fasttext_model.wv[word]

In [ ]:
# FastText Dataset
train_dataset_fasttext = TextEmbeddingDataset(train_inputs, train_targets, word2idx_fasttext, max_len)
test_dataset_fasttext = TextEmbeddingDataset(test_inputs, test_targets, word2idx_fasttext, max_len)

# FastText DataLoader
train_loader_fasttext = DataLoader(train_dataset_fasttext, batch_size=64, shuffle=True)
test_loader_fasttext = DataLoader(test_dataset_fasttext, batch_size=64, shuffle=False)

In [ ]:
# model_fasttext = EmbeddingLSTM(fasttext_matrix, hidden_dim, output_dim).to(device)

model_fasttext = EmbeddingGRU(fasttext_matrix, hidden_dim=128, output_dim=20).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_fasttext.parameters(), lr=0.005)

for epoch in range(10):
    loss = train(model_fasttext, train_loader_fasttext, loss_fn, optimizer)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

accuracy = evaluate(model_fasttext, test_loader_fasttext)
print(f"Test Accuracy (FastText with LSTM): {accuracy:.4f}")


# 결과
gru = 0.692
- word2vec이랑 비교해봐도 뭐 또이또이함, test loss 떨어지는 속도도 비슷하고... 트레인에 오버피팅인듯